# CeNN Adaptive Routing + MaxPool Attention — Colab ablation

This experiment starts from the previous winner `cellular_multiscale5` and tests three targeted upgrades:

- `cellular_adaptive_multiscale5`: token/head-specific learned routing over `{0, 1, d, 2d, 4d}`.
- `cellular_multiscale5_maxpool`: the same sparse attention plus an old-school channel-wise MaxPool residual branch.
- `cellular_adaptive_maxpool5`: adaptive routing and MaxPool together.
- `cellular_multiscale5`: unchanged control from the previous experiment.

The scientific target is to reduce held-out `delta_nll` toward 0 while preserving the sparse score-pair ratio. Validation selects the winner; the held-out test set is not used for architecture selection.


## 1 · Fresh checkout and dependencies


In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REPO_REF = "main"
WORK_PARENT = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
REPO_DIR = pathlib.Path(tempfile.mkdtemp(prefix="TinyCeNN-adaptive-maxpool-", dir=WORK_PARENT))

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(REPO_DIR),
    "transformers==4.57.6", "datasets>=3,<5", "huggingface_hub>=0.34,<2",
    "pandas", "matplotlib", "pytest>=8"
], check=True)
for path in (REPO_DIR, REPO_DIR / "src"):
    sys.path.insert(0, str(path))
importlib.invalidate_caches()
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Python:", sys.version)
print("Source commit:", SOURCE_COMMIT)
print("Checkout:", REPO_DIR)


## 2 · Experiment configuration


In [ ]:
import torch
from pathlib import Path

PROFILE = "balanced"      # smoke | balanced | extended
LAYERS = "18"
SEED = 2026
FEATURE_DIMS = "32,64,96"
ALLOW_CPU = False
SAVE_TO_DRIVE = False
AUTO_DOWNLOAD = True

# Put a new path first so the preflight exercises the new implementation.
VARIANTS = (
    "cellular_adaptive_multiscale5,"
    "cellular_adaptive_maxpool5,"
    "cellular_multiscale5_maxpool,"
    "cellular_multiscale5"
)

if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("GPU not detected. In Colab choose Runtime -> Change runtime type -> GPU.")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total:.1f} GiB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_ROOT = Path("/content/drive/MyDrive/TinyCeNN-LM/adaptive-maxpool-attention")
else:
    RESULT_ROOT = WORK_PARENT / "TinyCeNN-adaptive-maxpool-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Profile:", PROFILE)
print("Variants:", VARIANTS)
print("Feature dims:", FEATURE_DIMS)
print("Result root:", RESULT_ROOT)


## 3 · Unit/regression tests


In [ ]:
test_command = [
    sys.executable, "-m", "pytest", "-q",
    str(REPO_DIR / "tests/test_cellular_attention.py"),
    str(REPO_DIR / "tests/test_research_layer_benchmark.py"),
    str(REPO_DIR / "tests/test_cellular_attention_cli.py"),
]
print("Running:", subprocess.list2cmdline(test_command))
subprocess.run(test_command, cwd=REPO_DIR, check=True)
print("✅ Cellular Attention tests passed")


## 4 · Real-model preflight + balanced ablation


In [ ]:
import os

runner = REPO_DIR / "scripts/run_cellular_attention_colab.py"
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["HF_HUB_DISABLE_XET"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

command = [
    sys.executable, "-u", str(runner),
    "--profile", PROFILE,
    "--layers", LAYERS,
    "--variants", VARIANTS,
    "--feature-dims", FEATURE_DIMS,
    "--seed", str(SEED),
    "--output-root", str(RESULT_ROOT),
]
print("Command:", subprocess.list2cmdline(command))
result = subprocess.run(command, cwd=REPO_DIR, env=env)
RUN_RETURN_CODE = result.returncode
print("Launcher exit code:", RUN_RETURN_CODE)


## 5 · Inspect status and load results


In [ ]:
import json
import pandas as pd
from IPython.display import display

STATUS_FILE = RESULT_ROOT / "last_run.json"
if not STATUS_FILE.exists():
    raise RuntimeError(f"Launcher did not create {STATUS_FILE}")
status = json.loads(STATUS_FILE.read_text(encoding="utf-8"))
print(json.dumps(status, indent=2))
if status["status"] != "completed":
    if status.get("error_tail"):
        print("\nLast log lines:\n", status["error_tail"])
    raise RuntimeError(f"Benchmark failed at stage: {status['status']}")

OUTPUT_DIR = Path(status["output_dir"])
LOG_FILE = Path(status["log_file"])
report = json.loads((OUTPUT_DIR / "cellular_attention_report.json").read_text())
results = pd.read_csv(OUTPUT_DIR / "cellular_attention_summary.csv")
validation = pd.read_csv(OUTPUT_DIR / "validation_summary.csv")
print("✅ Benchmark completed")
print("Results:", OUTPUT_DIR)
print("Validation winner:", report["validation_winners"])


## 6 · Which idea wins?


In [ ]:
candidates = results[results["variant"] != "transformer_original"].copy()
selected = candidates[candidates["selected_on_validation"].eq(True)].copy()

print("Validation ranking (lower NLL is better):")
display(validation.sort_values("validation_nll")[[
    "variant", "feature_dim", "validation_nll", "validation_delta_nll",
    "validation_output_cosine", "trainable_parameters", "receptive_field_tokens"
]].round(6))

print("\nHeld-out result for validation-selected winner:")
display(selected[[
    "candidate", "variant", "feature_dim", "context",
    "test_nll", "transformer_perplexity", "test_perplexity", "ppl_ratio",
    "delta_nll", "delta_nll_ci_low", "delta_nll_ci_high",
    "score_pair_ratio", "prefill_speedup", "strict_quality_win", "quality"
]].sort_values("context").round(6))

print("\nAll candidates at each context, best first:")
display(candidates[[
    "variant", "feature_dim", "context", "selected_on_validation",
    "test_perplexity", "ppl_ratio", "delta_nll", "delta_nll_ci_high",
    "score_pair_ratio", "output_cosine", "grad_mean_cosine", "quality"
]].sort_values(["context", "delta_nll"]).round(6))


## 7 · Quality vs sparsity


In [ ]:
import matplotlib.pyplot as plt

for context in sorted(candidates["context"].unique()):
    view = candidates[candidates["context"].eq(context)].copy()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(view["score_pair_ratio"], view["delta_nll"], s=90)
    for _, row in view.iterrows():
        ax.annotate(
            f"{row['variant']} f{int(row['feature_dim'])}",
            (row["score_pair_ratio"], row["delta_nll"]),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
    ax.axhline(0, linewidth=1)
    ax.axhline(0.02, linewidth=1, linestyle="--")
    ax.set_title(f"Adaptive/MaxPool Cellular Attention — context {context}")
    ax.set_xlabel("Sparse / dense score-pair ratio (lower is better)")
    ax.set_ylabel("Delta NLL vs Transformer (lower is better)")
    ax.grid(True, alpha=0.2)
    plt.show()


## 8 · Package and download results


In [ ]:
import shutil

ZIP_BASE = RESULT_ROOT / f"adaptive-maxpool-{OUTPUT_DIR.name}"
ZIP_PATH = Path(shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUTPUT_DIR))
print("ZIP:", ZIP_PATH)

if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(ZIP_PATH))
        print("✅ Download triggered")
    except Exception as exc:
        print("Automatic download unavailable:", exc)

print("Keep this ZIP; it contains the manifest, checkpoints, validation selection, held-out NLL, and full report.")
